In [ ]:

#####################################################################
# STEP 1: IMPORTING LIBRARIES & HARDWARE CHECK
#####################################################################
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
os.environ['NCCL_DEBUG'] = 'WARN'
print("[INFO] Loading required python libraries...")
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
from glob import glob
from PIL import Image

import tensorflow as tf
print("\n" + "="*50)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("Name: ", tf.config.list_physical_devices('GPU'))
print("="*50 + "\n")

from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Recall, Precision
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, LeakyReLU, Add, Multiply
from tensorflow.keras.models import Model

#####################################################################
# STEP 2: PROJECT CONSTANTS
#####################################################################
H = 256
W = 256
BATCH_SIZE_PER_REPLICA = 8 
LEARNING_RATE = 1e-4
EPOCHS = 150  
LR_PATIENCE = 5
ES_PATIENCE = 10
SMOOTH = 1e-6

def combo_loss(y_true, y_pred):
    def dice_coef(y_t, y_p):
        it = tf.reduce_sum(y_t * y_p)
        return (2. * it + SMOOTH) / (tf.reduce_sum(y_t) + tf.reduce_sum(y_p) + SMOOTH)
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = 1.0 - dice_coef(y_true, y_pred)
    return bce + dice

#####################################################################
# STEP 3: DATA PIPELINE (PAPER DATASET NESTED - 80/10/10 SPLIT)
#####################################################################
def load_data(base_path):
    """Load paper dataset (nested per-sample folders, pre-split Train/Test)."""
    train_dir = os.path.join(base_path, "Training", "training")
    test_dir  = os.path.join(base_path, "Testing", "testing")
    print(f"[INFO] Training dir: {train_dir}")
    print(f"[INFO] Testing dir:  {test_dir}")

    if not os.path.exists(train_dir):
        print(f"[ERROR] Training directory not found: {train_dir}")
        return ([], []), ([], []), ([], [])

    # Each sample is in its own subfolder: <sample>/images/*.png, <sample>/masks/*.png
    images = sorted(glob(os.path.join(train_dir, "*", "images", "*.png")))
    masks  = sorted(glob(os.path.join(train_dir, "*", "masks", "*.png")))

    test_images = sorted(glob(os.path.join(test_dir, "*", "images", "*.png")))
    test_masks  = sorted(glob(os.path.join(test_dir, "*", "masks", "*.png")))

    if not images:
        print("[ERROR] No training images found!")
        return ([], []), ([], []), ([], [])

    assert len(images) == len(masks), f"Train alignment: {len(images)} imgs vs {len(masks)} masks"
    assert len(test_images) == len(test_masks), f"Test alignment: {len(test_images)} vs {len(test_masks)}"
    print(f"[INFO] Found {len(images)} training + {len(test_images)} testing image-mask pairs.")

    # Split training into train + val (80/20); test comes from Testing folder
    tx, vx, ty, vy = train_test_split(images, masks, test_size=0.2, random_state=42)
    return (tx, ty), (vx, vy), (test_images, test_masks)

def read_image(path):
    try:
        img = Image.open(path).convert('RGB').resize((W, H))
        return np.array(img, dtype=np.float32) / 255.0
    except: return None

def read_mask(path):
    try:
        mask = Image.open(path).convert('L').resize((W, H))
        return np.expand_dims(np.array(mask, dtype=np.float32) / 255.0, axis=-1)
    except: return None

def tf_parse(x, y):
    def _p(x, y): return read_image(x), read_mask(y)
    x, y = tf.numpy_function(_p, [x, y], [tf.float32, tf.float32])
    x.set_shape([H, W, 3]); y.set_shape([H, W, 1])
    return x, y

def heavy_augment(x, y):
    if tf.random.uniform(()) > 0.5: x = tf.image.flip_left_right(x); y = tf.image.flip_left_right(y)
    if tf.random.uniform(()) > 0.5: x = tf.image.flip_up_down(x); y = tf.image.flip_up_down(y)
    x = tf.image.random_brightness(x, 0.1)
    x = tf.image.random_contrast(x, 0.9, 1.1)
    x = tf.clip_by_value(x, 0.0, 1.0)
    return x, y

#####################################################################
# STEP 4: MODEL ARCHITECTURE (ATTENTION-GUIDED RES-UNET 34)
#####################################################################
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    x = Conv2D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    return x

def residual_block(x, filters):
    res = Conv2D(filters, (1, 1), padding='same')(x)
    res = BatchNormalization()(res)
    x = conv_block(x, filters)
    x = Add()([x, res])
    x = LeakyReLU(negative_slope=0.1)(x)
    return x

def attention_gate(x, g, inter_filters):
    """Attention Gate -- suppress irrelevant features in skip connections.
    x: encoder skip features, g: decoder gating signal, inter_filters: bottleneck channels.
    Returns: x weighted by learned spatial attention coefficients."""
    Wg = Conv2D(inter_filters, (1, 1), padding='same')(g)
    Wg = BatchNormalization()(Wg)
    Wx = Conv2D(inter_filters, (1, 1), padding='same')(x)
    Wx = BatchNormalization()(Wx)
    psi = Add()([Wg, Wx])
    psi = LeakyReLU(negative_slope=0.1)(psi)
    psi = Conv2D(1, (1, 1), padding='same', activation='sigmoid')(psi)
    return Multiply()([x, psi])

def build_resnet(input_shape=(256, 256, 3)):
    inputs = Input(input_shape)
    c1 = residual_block(inputs, 64); p1 = MaxPool2D((2, 2))(c1)
    c2 = residual_block(p1, 128); p2 = MaxPool2D((2, 2))(c2)
    c3 = residual_block(p2, 256); p3 = MaxPool2D((2, 2))(c3)
    c4 = residual_block(p3, 512); p4 = MaxPool2D((2, 2))(c4)
    bn = residual_block(p4, 1024)
    d1 = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(bn)
    c4_att = attention_gate(c4, d1, 256)
    d1 = Concatenate()([d1, c4_att]); d1 = residual_block(d1, 512)
    d2 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(d1)
    c3_att = attention_gate(c3, d2, 128)
    d2 = Concatenate()([d2, c3_att]); d2 = residual_block(d2, 256)
    d3 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(d2)
    c2_att = attention_gate(c2, d3, 64)
    d3 = Concatenate()([d3, c2_att]); d3 = residual_block(d3, 128)
    d4 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(d3)
    c1_att = attention_gate(c1, d4, 32)
    d4 = Concatenate()([d4, c1_att]); d4 = residual_block(d4, 64)
    outputs = Conv2D(1, (1, 1), padding='same', activation='sigmoid')(d4)
    return Model(inputs, outputs)

#####################################################################
# STEP 5: TRAINING THE MODEL (PAPER DATASET - KAGGLE)
#####################################################################
DATASETS = [{"name": "PaperDataset", "path": "/kaggle/input/datasets/bandatharun/road-detection-satellite-tiles-equatorial-asia"}]
for ds in DATASETS:
    print(f"\n{'='*50}\n TRAINING CYCLE: {ds['name']}\n{'='*50}")
    
    # ---------------------------------------------------------
    # 1. LOAD DATA
    # ---------------------------------------------------------
    (train_x, train_y), (val_x, val_y), (test_x, test_y) = load_data(ds['path'])
    if not train_x: continue

    # ---------------------------------------------------------
    # 2. CONSTRUCT TF_DATASET (DUAL-GPU TARGETING)
    # ---------------------------------------------------------
    strategy = tf.distribute.MirroredStrategy()
    GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
    
    print(f"[INFO] Using Global Batch Size: {GLOBAL_BATCH_SIZE}")
    train_dataset = tf.data.Dataset.from_tensor_slices((train_x, train_y)).map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
    if True:
        print("[INFO] Data Augmentation is ENABLED.")
        train_dataset = train_dataset.map(heavy_augment, num_parallel_calls=tf.data.AUTOTUNE)
    
    train_dataset = train_dataset.shuffle(buffer_size=500).repeat().batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_dataset = tf.data.Dataset.from_tensor_slices((val_x, val_y)).map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE).batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    test_dataset = tf.data.Dataset.from_tensor_slices((test_x, test_y)).map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE).batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    options = tf.data.Options()
    options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
    train_dataset = train_dataset.with_options(options)
    val_dataset = val_dataset.with_options(options)
    test_dataset = test_dataset.with_options(options)

    # ---------------------------------------------------------
    # 3. BUILD & COMPILE MODEL WITH GPU MIRRORED SCOPE
    # ---------------------------------------------------------
    with strategy.scope():
        model = build_resnet(input_shape=(256, 256, 3))

        def iou(y_true, y_pred):
            y_pred = tf.cast(y_pred > 0.5, tf.float32)
            intersection = tf.reduce_sum(y_true * y_pred)
            union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
            return (intersection + SMOOTH) / (union + SMOOTH)

        def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75):
            '''
            Focal Tversky Loss: Advanced Novelty for this research project.
            Heavily penalizes missed faint pixels (False Negatives).
            '''
            y_true = tf.cast(y_true, tf.float32)
            y_pred = tf.cast(y_pred, tf.float32)
            tp = tf.reduce_sum(y_true * y_pred)
            fn = tf.reduce_sum(y_true * (1 - y_pred))
            fp = tf.reduce_sum((1 - y_true) * y_pred)
            tversky_index = (tp + SMOOTH) / (tp + alpha * fn + beta * fp + SMOOTH)
            return tf.pow((1 - tversky_index), gamma)

        print("[INFO] Compiling model inside GPU mapping strategy...")
        def connectivity_penalty(y_true, y_pred):
            """Differentiable connectivity loss via Laplacian edge matching.
            Penalizes structural discontinuities in predicted road masks."""
            laplacian_kernel = tf.constant([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=tf.float32)
            laplacian_kernel = tf.reshape(laplacian_kernel, [3, 3, 1, 1])
            edges_pred = tf.nn.conv2d(y_pred, laplacian_kernel, strides=[1,1,1,1], padding='SAME')
            edges_true = tf.nn.conv2d(y_true, laplacian_kernel, strides=[1,1,1,1], padding='SAME')
            return tf.reduce_mean(tf.abs(edges_pred - edges_true))

        def proposed_loss(y_true, y_pred):
            """Novel Combined Loss: Focal Tversky + Connectivity Penalty (lambda=0.3).
            Focal Tversky handles class imbalance; connectivity penalty preserves road structure."""
            ftl = focal_tversky_loss(y_true, y_pred)
            conn = connectivity_penalty(y_true, y_pred)
            return ftl + 0.3 * conn

        model.compile(loss=proposed_loss, optimizer=Adam(LEARNING_RATE), metrics=[iou, Precision(), Recall()])
    
    model.summary()
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_iou', mode='max', patience=ES_PATIENCE, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_iou', factor=0.5, patience=LR_PATIENCE, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(filepath=f"best_model_proposed.keras", monitor='val_iou', mode='max', save_best_only=True, verbose=1)
    ]

    # ---------------------------------------------------------
    # 4. TRAIN THE MODEL
    # ---------------------------------------------------------
    print("[INFO] Starting training loop...")
    history = model.fit(train_dataset, epochs=EPOCHS, steps_per_epoch=np.ceil(len(train_x)/GLOBAL_BATCH_SIZE).astype(int), validation_data=val_dataset, callbacks=callbacks)
    
    # ---------------------------------------------------------
    # 5. SAVE WEIGHTS
    # ---------------------------------------------------------
    model.save(f"final_model_proposed.keras")
    print(f"[SUCCESS] Model saved to final_model_proposed.keras")

    print("\n[INFO] Evaluating on held-out test set...")
    test_results = model.evaluate(test_dataset, verbose=1)
    print(f"Test Set Results: {test_results}")

    # ---------------------------------------------------------
    # 6. PLOT LOSS CURVES
    # ---------------------------------------------------------
    print("[INFO] Plotting performance metrics...")
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.plot(history.history['loss'], label='Train'); plt.plot(history.history['val_loss'], label='Val'); plt.title('Loss Curve'); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(history.history['iou'], label='Train'); plt.plot(history.history['val_iou'], label='Val'); plt.title('IoU Curve'); plt.legend()
    plt.show()
    
    # NOVEL POST-PROCESSING: Flood-Fill & Morphology
    print("[INFO] Running novel post-processing on a test sample.")
    sample_img = read_image(test_x[0])
    if sample_img is not None:
        p = model.predict(np.expand_dims(sample_img, 0))[0]
        m = (p.squeeze() > 0.5).astype(np.uint8) * 255
        clean = m.copy()
        gray = cv2.cvtColor(np.uint8(sample_img*255), cv2.COLOR_RGB2GRAY)
        b_mask = (gray == 0).astype(np.uint8)
        # Seed at corners (x, y) order
        for s in [(0,0), (255,0), (0,255), (255,255)]:
            f_mask = np.zeros((258, 258), np.uint8)
            if b_mask[s[1], s[0]] == 1: cv2.floodFill(clean, f_mask, s, 0)
        # Morphological Closing
        clean = cv2.morphologyEx(clean, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))
        plt.figure(figsize=(12, 4))
        plt.subplot(1,3,1); plt.imshow(sample_img); plt.title('Input')
        plt.subplot(1,3,2); plt.imshow(m, cmap='gray'); plt.title('Raw AI')
        plt.subplot(1,3,3); plt.imshow(clean, cmap='gray'); plt.title('Proposed')
        plt.show()
    #####################################################################
    # STEP 7: NOVEL CONNECTIVITY METRIC -- FULL TEST SET EVALUATION
    #####################################################################
    print("\n" + "="*50)
    print(" CONNECTIVITY METRIC -- FULL TEST SET")
    print("="*50)

    def connectivity_score(pred_bin, true_bin):
        """Novel metric: ratio of connected components (GT / Pred).
        Near 1.0 = well-connected roads. << 1.0 = fragmented prediction."""
        _, n_pred = cv2.connectedComponents(pred_bin)
        _, n_true = cv2.connectedComponents(true_bin)
        return n_true / max(n_pred, 1)

    conn_scores, iou_np = [], []
    num_eval = min(len(test_x), 50)
    print("[INFO] Evaluating connectivity on " + str(num_eval) + " samples...")
    for idx in range(num_eval):
        t_img = read_image(test_x[idx])
        t_msk = read_mask(test_y[idx])
        if t_img is None or t_msk is None: continue
        t_pred = model.predict(np.expand_dims(t_img, 0), verbose=0)[0]
        pb = (t_pred.squeeze() > 0.5).astype(np.uint8)
        tb = (t_msk.squeeze() > 0.5).astype(np.uint8)
        inter = np.sum(pb * tb)
        union = np.sum(pb) + np.sum(tb) - inter
        iou_np.append(float((inter + 1e-6) / (union + 1e-6)))
        conn_scores.append(connectivity_score(pb, tb))

    print("\n-- Connectivity Analysis (" + str(len(conn_scores)) + " samples) --")
    print("  Mean IoU:          " + str(round(float(np.mean(iou_np)), 4)))
    print("  Mean Connectivity: " + str(round(float(np.mean(conn_scores)), 4)))
    print("  Std  Connectivity: " + str(round(float(np.std(conn_scores)), 4)))
